# 02c m9_pbm Training-Regime Comparison

**Research question.** Does adding simulated Alpha data improve the
substation-held-out performance of the compact physical model on Beta?

This notebook compares three threshold-training regimes while holding the
candidate score fixed as the equal-weight mean of F1 bridge improvement, F3
slope-continuity improvement, and F4 duration plausibility. Each model selects
its own highest-scoring candidate window before the day threshold is applied.

**Inputs:** the 02b candidate and day caches.  
**Outputs:** fold thresholds, day metrics, two compact tables, two figures, and
a reproducibility manifest.  
**Expected runtime:** under five minutes after 02b exists.

## 1. Imports, Paths, And Compact-Model Definition

This experiment uses equal weights only. It does not optimise feature weights
and does not train a machine-learning classifier. Those questions are isolated
in later notebooks.

In [1]:
from pathlib import Path
import sys
import time

import pandas as pd
from IPython.display import display


def find_notebook_directory() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if candidate.name == "notebooks" and (candidate / "_m9_pbm_data.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article" / "notebooks"
        if (nested / "_m9_pbm_data.py").exists():
            return nested
    raise FileNotFoundError("Could not locate the journal notebook directory.")


NOTEBOOK_DIR = find_notebook_directory()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _m9_pbm_data import (  # noqa: E402
    load_experiment_config,
    manifest_payload,
    output_dirs,
    resolve_paths,
    write_csv,
    write_manifest,
    write_parquet,
)
from _m9_pbm_features import (  # noqa: E402
    COMPACT_FEATURE_COLUMNS,
    select_best_candidates,
)
from _m9_pbm_plotting import (  # noqa: E402
    plot_regime_metrics,
    plot_regime_thresholds,
)
from _m9_pbm_validation import (  # noqa: E402
    assert_heldout_absent,
    equal_weights,
    metric_rows,
    select_threshold,
)

STARTED_AT = time.time()
ARTICLE_ROOT = NOTEBOOK_DIR.parent
CONFIG = load_experiment_config(ARTICLE_ROOT)
PATHS = resolve_paths(ARTICLE_ROOT, CONFIG)
SLUG = "02c_m9_pbm_training_regimes"
OUTPUT_DIRS = output_dirs(PATHS, SLUG)
WEIGHTS = equal_weights(COMPACT_FEATURE_COLUMNS)

display(pd.Series(WEIGHTS, name="equal_candidate_weights"))
display(pd.Series(CONFIG["m9_pbm"]["threshold_selection"], name="threshold_selection"))

F1_bridge_improvement              0.333333
F3_slope_continuity_improvement    0.333333
F4_duration_plausibility           0.333333
Name: equal_candidate_weights, dtype: float64

objective                                   macro_substation_f1
tie_breaks    [macro_precision, macro_recall, higher_threshold]
Name: threshold_selection, dtype: object

## 2. Score, Window Selection, And Day Threshold

For candidate window $W$,

$$
Score_{equal}(W)=\frac{F_1(W)+F_3(W)+F_4(W)}{3}.
$$

The selected window and day prediction are

$$
W_d^*=\operatorname*{arg\,max}_{W}Score_{equal}(W),
\qquad
\widehat{RPF}(d)=\mathbb{1}\{Score_{equal}(W_d^*)\geq\tau\}.
$$

**Notation**

| Symbol | Meaning |
|---|---|
| $W$ | A valid 30-minute to 8-hour candidate window. |
| $d$ | One substation-day. |
| $F_1(W)$ | Bridge-improvement score for $W$. |
| $F_3(W)$ | Slope-continuity-improvement score for $W$. |
| $F_4(W)$ | Duration-plausibility score for $W$. |
| $Score_{equal}(W)$ | Equal-weight compact physical score. |
| $W_d^*$ | Highest-scoring candidate on day $d$. |
| $\tau$ | Threshold selected from training substations only. |

Candidate selection is not classification: every day has a best candidate, but
only a best-candidate score at or above $\tau$ produces a positive RPF day.

## 3. Reduce The Candidate Cache Once

The 9.77-million-row cache is processed one substation partition at a time. The
result has one selected candidate per Alpha/Beta substation-day. Labels and Beta
confidence are joined only after physical scoring, with a one-to-one assertion.

In [2]:
CACHE_ROOT = PATHS.intermediate / "02b_m9_pbm_candidate_features"
PARTITION_DIR = CACHE_ROOT / "_partitions"
DAY_INPUT_CACHE = CACHE_ROOT / "day_input_cache.parquet"
DAILY_CACHE = OUTPUT_DIRS["intermediate"] / "compact_equal_daily_candidates.parquet"

required_inputs = [PARTITION_DIR, DAY_INPUT_CACHE]
assert all(path.exists() for path in required_inputs), "Run Notebook 02b first."

if DAILY_CACHE.exists() and CONFIG["execution"]["resume_validated_intermediates"]:
    daily = pd.read_parquet(DAILY_CACHE)
else:
    selected_parts = []
    candidate_paths = sorted(PARTITION_DIR.glob("*_candidates.parquet"))
    assert len(candidate_paths) == 18
    for candidate_path in candidate_paths:
        candidates = pd.read_parquet(candidate_path)
        selected_parts.append(select_best_candidates(candidates, WEIGHTS))
    selected = pd.concat(selected_parts, ignore_index=True)
    labels = pd.read_parquet(DAY_INPUT_CACHE)
    daily = selected.merge(
        labels[
            [
                "dataset",
                "substation_id",
                "date",
                "true_day",
                "true_interval_count",
                "confidence",
            ]
        ],
        on=["dataset", "substation_id", "date"],
        how="left",
        validate="one_to_one",
    )
    assert daily["true_day"].notna().all()
    write_parquet(daily, DAILY_CACHE)

assert len(daily) == sum(CONFIG["datasets"]["expected_substation_days"].values())
assert daily.duplicated(["dataset", "substation_id", "date"]).sum() == 0
display(
    daily.groupby("dataset", as_index=False).agg(
        substation_days=("date", "size"),
        positive_days=("true_day", "sum"),
        substations=("substation_id", "nunique"),
        mean_selected_score=("score", "mean"),
    )
)

,dataset,substation_days,positive_days,substations,mean_selected_score
0,alpha,10643,3423,10,0.413216
1,beta,2928,630,8,0.433735


## 4. Three Training Regimes And Leakage Controls

**Beta only.** Hold out one Beta substation. Select the threshold using sure
days from the other seven Beta substations, then predict every day from the
held-out substation.

**Beta plus Alpha.** Use the same held-out Beta fold, but add all Alpha days to
the seven-substation Beta training set. Alpha and Beta receive equal total
influence in the macro threshold objective, so Alpha's larger sample cannot
dominate.

**Alpha only.** Select one threshold using Alpha alone, then transfer it to all
eight Beta substations. No Beta label contributes to this threshold.

For every regime, Beta sure and Beta all are reporting scopes rather than
alternative predictions. Held-out confidence is consulted only after the
predictions exist.

In [3]:
alpha = daily.loc[daily["dataset"].eq("alpha")].copy()
beta = daily.loc[daily["dataset"].eq("beta")].copy()
beta_substations = sorted(beta["substation_id"].unique())

threshold_rows = []
prediction_parts = []
for regime in ["beta_only", "beta_plus_alpha"]:
    for heldout_substation in beta_substations:
        beta_training = beta.loc[
            beta["confidence"].eq("sure")
            & ~beta["substation_id"].eq(heldout_substation)
        ].copy()
        assert_heldout_absent(beta_training, heldout_substation)
        if regime == "beta_only":
            training = beta_training
            dataset_balanced = False
        else:
            training = pd.concat([alpha, beta_training], ignore_index=True)
            dataset_balanced = True

        selection = select_threshold(
            training,
            score_column="score",
            dataset_balanced=dataset_balanced,
        )
        evaluation = beta.loc[beta["substation_id"].eq(heldout_substation)].copy()
        evaluation["regime"] = regime
        evaluation["heldout_substation"] = heldout_substation
        evaluation["threshold"] = selection.threshold
        evaluation["predicted_day"] = evaluation["score"].ge(selection.threshold)
        prediction_parts.append(evaluation)
        threshold_rows.append(
            {
                "regime": regime,
                "heldout_substation": heldout_substation,
                "training_alpha_substations": 0 if regime == "beta_only" else 10,
                "training_beta_substations": 7,
                "training_beta_confidence": "sure_only",
                "dataset_balanced": dataset_balanced,
                "training_rows": len(training),
                "threshold": selection.threshold,
                **selection.metrics,
            }
        )

# Pure Alpha-to-Beta transfer uses no Beta label in threshold selection.
alpha_selection = select_threshold(alpha, score_column="score", dataset_balanced=False)
alpha_transfer = beta.copy()
alpha_transfer["regime"] = "alpha_only"
alpha_transfer["heldout_substation"] = alpha_transfer["substation_id"]
alpha_transfer["threshold"] = alpha_selection.threshold
alpha_transfer["predicted_day"] = alpha_transfer["score"].ge(alpha_selection.threshold)
prediction_parts.append(alpha_transfer)
threshold_rows.append(
    {
        "regime": "alpha_only",
        "heldout_substation": "all_beta",
        "training_alpha_substations": 10,
        "training_beta_substations": 0,
        "training_beta_confidence": "not_used",
        "dataset_balanced": False,
        "training_rows": len(alpha),
        "threshold": alpha_selection.threshold,
        **alpha_selection.metrics,
    }
)

predictions = pd.concat(prediction_parts, ignore_index=True)
thresholds = pd.DataFrame(threshold_rows)
assert predictions.groupby("regime").size().eq(len(beta)).all()
assert predictions.groupby(["regime", "substation_id", "date"]).size().eq(1).all()
display(thresholds[["regime", "heldout_substation", "training_rows", "threshold", "macro_f1"]])

,regime,heldout_substation,training_rows,threshold,macro_f1
0,beta_only,beta_A,1973,0.534262,0.659303
1,beta_only,beta_B,2079,0.600687,0.648295
2,beta_only,beta_C,1971,0.599893,0.767561
3,beta_only,beta_D,2122,0.599893,0.655411
4,beta_only,beta_E,2002,0.599893,0.641315
5,beta_only,beta_F,2023,0.599893,0.634512
6,beta_only,beta_G,2027,0.599893,0.656243
7,beta_only,beta_H,1973,0.599893,0.719942
8,beta_plus_alpha,beta_A,12616,0.528792,0.624784
9,beta_plus_alpha,beta_B,12722,0.528792,0.604705


## 5. Held-Out Beta Metrics

The paper headline is pooled precision, recall, and F1 on held-out **Beta sure**
days. Beta all is retained as a secondary sensitivity analysis. Macro-substation
rows average the eight independently computed substation metrics and are shown
beside pooled results so one large substation cannot hide poor transfer.

In [4]:
metric_parts = []
for regime, regime_frame in predictions.groupby("regime", sort=False):
    for confidence_scope, evaluation in [
        ("beta_sure", regime_frame.loc[regime_frame["confidence"].eq("sure")]),
        ("beta_all", regime_frame),
    ]:
        rows = metric_rows(evaluation)
        rows.insert(0, "confidence_scope", confidence_scope)
        rows.insert(0, "regime", regime)
        metric_parts.append(rows)
day_metrics = pd.concat(metric_parts, ignore_index=True)

headline = day_metrics.loc[
    day_metrics["confidence_scope"].eq("beta_sure")
    & day_metrics["aggregation"].isin(["pooled", "macro_substation"])
].copy()
by_substation = day_metrics.loc[day_metrics["aggregation"].eq("substation")].copy()

METRICS_PATH = OUTPUT_DIRS["metrics"] / "01_day_metrics.csv"
THRESHOLDS_PATH = OUTPUT_DIRS["metrics"] / "02_thresholds_by_fold.csv"
HEADLINE_PATH = OUTPUT_DIRS["tables"] / "table01_regime_headline_metrics.csv"
SUBSTATION_PATH = OUTPUT_DIRS["tables"] / "table02_regime_metrics_by_substation.csv"
PREDICTIONS_PATH = OUTPUT_DIRS["intermediate"] / "daily_regime_predictions.parquet"

write_csv(day_metrics, METRICS_PATH)
write_csv(thresholds, THRESHOLDS_PATH)
write_csv(headline, HEADLINE_PATH)
write_csv(by_substation, SUBSTATION_PATH)
write_parquet(predictions, PREDICTIONS_PATH)

display(headline[["regime", "aggregation", "support", "precision", "recall", "f1"]])

,regime,aggregation,support,precision,recall,f1
0,beta_only,pooled,2310,0.832981,0.836518,0.834746
9,beta_only,macro_substation,2310,0.658852,0.682429,0.647038
20,beta_plus_alpha,pooled,2310,0.773619,0.921444,0.841085
29,beta_plus_alpha,macro_substation,2310,0.569501,0.740795,0.629774
40,alpha_only,pooled,2310,0.669578,0.976645,0.794473
49,alpha_only,macro_substation,2310,0.494322,0.831639,0.596974


## 6. Figures

The first figure compares held-out pooled Beta-sure scores. The second exposes
fold-to-fold threshold variation for the two Beta LOSO regimes; Alpha-only has
one transfer threshold and is therefore reported in the threshold table rather
than as a misleading eight-point line.

In [5]:
FIGURE_METRICS = OUTPUT_DIRS["figures"] / "fig01_regime_precision_recall_f1.png"
FIGURE_THRESHOLDS = OUTPUT_DIRS["figures"] / "fig02_thresholds_by_heldout_substation.png"
pooled_sure = headline.loc[headline["aggregation"].eq("pooled")]
plot_regime_metrics(pooled_sure, FIGURE_METRICS)
plot_regime_thresholds(thresholds, FIGURE_THRESHOLDS)
display(FIGURE_METRICS)
display(FIGURE_THRESHOLDS)

WindowsPath('C:/Users/z5404477/Documents/PyNRPF/publication/2_journal_article/outputs/figures/02c_m9_pbm_training_regimes/fig01_regime_precision_recall_f1.png')

WindowsPath('C:/Users/z5404477/Documents/PyNRPF/publication/2_journal_article/outputs/figures/02c_m9_pbm_training_regimes/fig02_thresholds_by_heldout_substation.png')

## 7. Interpretation And Limitations

This comparison isolates the value of training data under one fixed compact
model. It does not claim that Beta is an untouched external dataset: feature
choice was informed by previous Beta development. LOSO nevertheless prevents
the held-out substation's labels from selecting its threshold.

The candidate cache is newly self-consistent, so these results may differ from
earlier fixed-window numbers. Later notebooks use the same candidate-selection
rule rather than forcing agreement with historical values.

In [6]:
MANIFEST_OUTPUTS = [
    METRICS_PATH,
    THRESHOLDS_PATH,
    HEADLINE_PATH,
    SUBSTATION_PATH,
    FIGURE_METRICS,
    FIGURE_THRESHOLDS,
]
manifest = manifest_payload(
    paths=PATHS,
    config=CONFIG,
    started_at=STARTED_AT,
    inputs=[
        PATHS.config,
        CACHE_ROOT / "candidate_feature_cache.parquet",
        DAY_INPUT_CACHE,
    ],
    outputs=MANIFEST_OUTPUTS,
    row_counts={
        "daily_selected_candidates": len(daily),
        "heldout_predictions_per_regime": len(beta),
        "regimes": predictions["regime"].nunique(),
        "threshold_rows": len(thresholds),
    },
)
manifest["local_intermediates"] = [
    str(DAILY_CACHE.relative_to(PATHS.article)),
    str(PREDICTIONS_PATH.relative_to(PATHS.article)),
]
MANIFEST_PATH = write_manifest(PATHS, f"{SLUG}.json", manifest)

inventory = pd.DataFrame(
    {"path": [*MANIFEST_OUTPUTS, PREDICTIONS_PATH, MANIFEST_PATH]}
)
inventory["exists"] = inventory["path"].map(Path.exists)
inventory["bytes"] = inventory["path"].map(lambda path: path.stat().st_size)
display(inventory)
assert inventory["exists"].all() and inventory["bytes"].gt(0).all()

,path,exists,bytes
0,C:\Users\z5404477\Documents\PyNRPF\publication...,True,6529
1,C:\Users\z5404477\Documents\PyNRPF\publication...,True,3695
2,C:\Users\z5404477\Documents\PyNRPF\publication...,True,813
3,C:\Users\z5404477\Documents\PyNRPF\publication...,True,5119
4,C:\Users\z5404477\Documents\PyNRPF\publication...,True,89543
5,C:\Users\z5404477\Documents\PyNRPF\publication...,True,111279
6,C:\Users\z5404477\Documents\PyNRPF\publication...,True,216671
7,C:\Users\z5404477\Documents\PyNRPF\publication...,True,2730
